# FARUMASI shift dispatcher — T4 sweep

Run this notebook from a fresh **T4** session.

1. **Runtime → Change runtime type → T4**.
2. If an older install already ran here, **Runtime → Restart session** first. That restores Colab's CUDA PyTorch.
3. **Runtime → Run all**.

The setup cell clones the repository and installs TorchRL without replacing Colab's PyTorch. Training then runs DQN, REINFORCE, PPO, and A2C: 10 settings each, 100,000 frames each, on the same pharmacy shift. On a T4 that takes about one to two hours. Running the training cell again skips experiments that already finished.

Leave this tab open. Do not save an older copy of the notebook back to GitHub.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), getattr(torch.version, "cuda", None))
if sys.version_info < (3, 12):
    raise RuntimeError(
        "TorchRL 0.15 needs Python 3.12 or newer. This runtime is Python "
        + sys.version.split()[0]
        + "."
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU in this session. Runtime → Change runtime type → T4, then Runtime → Restart session, then Runtime → Run all. "
        "Restart first if an older install already replaced Colab's CUDA PyTorch."
    )

REPO = "https://github.com/Emmanuel-NS/Rl_Mission-based_Summative.git"
base = Path("/content") if Path("/content").exists() else Path.cwd()
dest = base / "Rl_Mission-based_Summative"
if not (dest / "main.py").exists():
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
os.chdir(dest)
print("working directory", Path.cwd())

kept = torch.__version__
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "gymnasium>=1.0", "pygame>=2.6", "cloudpickle", "packaging", "pyvers",
    ]
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "torchrl==0.15.0", "tensordict==0.15.6", "hoptorch==0.1.4",
    ]
)

check = """
import torch
import gymnasium
import tensordict
import torchrl
from torchrl.collectors import Collector
from torchrl.objectives import A2CLoss, ClipPPOLoss, DQNLoss, ReinforceLoss
from training.torchrl_env import make_env
assert torch.cuda.is_available(), "CUDA torch was replaced: " + torch.__version__
env = make_env(0)
batch = env.reset()
print("ready", torch.__version__, "cuda True", "obs", tuple(batch.get("observation").shape))
"""
subprocess.check_call([sys.executable, "-c", check], cwd=dest)
print("setup kept torch", kept)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)
subprocess.check_call([sys.executable, "main.py", "--baselines"], cwd=Path.cwd())
subprocess.check_call([sys.executable, "main.py", "--screenshot"], cwd=Path.cwd())
print("screenshot", Path("assets/environment.png").resolve())


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)
print("Training 40 runs. Finished experiments are skipped if this cell is started again.")
subprocess.check_call(
    [
        sys.executable, "main.py",
        "--sweep", "all",
        "--device", "cuda",
        "--frames", "100000",
        "--eval-every", "10000",
    ],
    cwd=Path.cwd(),
)


In [ ]:
import zipfile
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if not (root / "main.py").exists():
    root = Path.cwd()
archive = Path("/content/farumasi_rl_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for folder in ("models", "logs", "assets"):
        base = root / folder
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if path.is_file() and path.name != ".gitkeep":
                handle.write(path, path.relative_to(root).as_posix())
print("saved", archive, "bytes", archive.stat().st_size)
try:
    from google.colab import files
    files.download(str(archive))
except Exception as error:
    print(error)
